### Week 2-3: Mortgage Rate Enrichment

In [10]:
import pandas as pd

sold = pd.read_csv('data/sold_cleaned.csv')
listings = pd.read_csv('data/listing_cleaned.csv')


url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

print(mortgage.head())
print(mortgage.dtypes)
print(mortgage.shape)


/var/folders/wj/4rpm0_z134d560x0595bhn900000gn/T/ipykernel_33382/977897868.py:3: DtypeWarning: Columns (16,29) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('data/sold_cleaned.csv')


        date  rate_30yr_fixed
0 1971-04-02             7.33
1 1971-04-09             7.31
2 1971-04-16             7.31
3 1971-04-23             7.31
4 1971-04-30             7.29
date               datetime64[ns]
rate_30yr_fixed           float64
dtype: object
(2872, 2)


In [5]:

mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
 mortgage.groupby('year_month')['rate_30yr_fixed']
 .mean()
 .reset_index()
)

print(mortgage_monthly.head(10))
print(mortgage_monthly.shape)
print(mortgage_monthly.dtypes)

  year_month  rate_30yr_fixed
0    1971-04           7.3100
1    1971-05           7.4250
2    1971-06           7.5300
3    1971-07           7.6040
4    1971-08           7.6975
5    1971-09           7.6875
6    1971-10           7.6280
7    1971-11           7.5500
8    1971-12           7.4800
9    1972-01           7.4375
(661, 2)
year_month         period[M]
rate_30yr_fixed      float64
dtype: object


In [ ]:
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
listings['year_month'] = pd.to_datetime(listings['ListingContractDate']).dt.to_period('M')


In [ ]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')


print(sold_with_rates.shape)
print(listings_with_rates.shape)

(262311, 50)
(400993, 47)
    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-09-12    2024-09   1090000.0             6.18
1  2024-09-30    2024-09   1995000.0             6.18
2  2024-09-30    2024-09   2340000.0             6.18
3  2024-09-30    2024-09    984000.0             6.18
4  2024-09-30    2024-09   1225000.0             6.18


In [17]:
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

print(
    sold_with_rates[
        ['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']
    ].head()
)


0
0
    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-09-12    2024-09   1090000.0             6.18
1  2024-09-30    2024-09   1995000.0             6.18
2  2024-09-30    2024-09   2340000.0             6.18
3  2024-09-30    2024-09    984000.0             6.18
4  2024-09-30    2024-09   1225000.0             6.18


In [19]:
sold_with_rates.to_csv('data/sold_with_rates.csv', index=False)
listings_with_rates.to_csv('data/listings_with_rates.csv', index=False)